# Build, Shape, and Extend an OpenClaw Agent on AMD
### From local agent basics to skills, customization, and a live adaptive app

---

**Your setup today:**
- AMD Instinct MI300X GPU (192 GB memory)
- Qwen3.5-122B model served via SGLang on `localhost:8090`
- OpenClaw agent framework
- A Python typing trainer app — your agent will explore it, find a bug, and fix it

**How this notebook works:**
- ▶ **Grey cells** → run them (Shift+Enter)
- 💬 **Prompt blocks** → copy-paste into the OpenClaw TUI in your terminal
- ⌨️ **Try it** → moments where you interact with the app or agent yourself

Keep a terminal open alongside this notebook.

---
## Section 0 — Verify Your Environment

The model server and OpenClaw are already installed. Let's confirm everything is up.

In [ ]:
import urllib.request, json, subprocess, time, pathlib, sys
sys.path.insert(0, ".")

try:
    req = urllib.request.Request(
        "http://localhost:8090/v1/models",
        headers={"Authorization": "Bearer abc-123"}
    )
    with urllib.request.urlopen(req) as r:
        models = json.loads(r.read())
    print("✅ SGLang is running")
    for m in models.get("data", []):
        print(f"   Model: {m['id']}")
except Exception as e:
    print(f"❌ SGLang not reachable: {e}")

In [ ]:
BASE_URL = "http://localhost:8090/v1"
API_KEY  = "abc-123"
MODEL_ID = "qwen3-5-122b"

subprocess.run(["openclaw", "config", "set", "gateway.mode", "local"], check=True)
subprocess.run(["openclaw", "config", "set", "agents.defaults.model", f"sglang/{MODEL_ID}"], check=True)

config_path = pathlib.Path.home() / ".openclaw" / "openclaw.json"
with config_path.open() as f:
    cfg = json.load(f)

cfg.setdefault("models", {}).setdefault("providers", {})["sglang"] = {
    "baseUrl": BASE_URL, "apiKey": API_KEY, "api": "openai-completions",
    "models": [{"id": MODEL_ID, "name": MODEL_ID, "reasoning": True,
                "input": ["text"], "cost": {"input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0},
                "contextWindow": 131072, "maxTokens": 8192}]
}
with config_path.open("w") as f:
    json.dump(cfg, f, indent=2)
print("✅ OpenClaw configured")

In [ ]:
log_file = pathlib.Path("/tmp/openclaw-gateway.log")
subprocess.run(["pkill", "-9", "-f", "openclaw-gateway"], capture_output=True)
time.sleep(1)
proc = subprocess.Popen(
    ["openclaw", "gateway", "run", "--bind", "loopback", "--port", "18789", "--force"],
    stdout=log_file.open("w"), stderr=subprocess.STDOUT,
)
time.sleep(4)
if proc.poll() is None:
    print(f"✅ Gateway running (PID {proc.pid})")
else:
    print("❌ Gateway failed:\n", log_file.read_text()[-600:])

---
## Section 1 — Meet Your Agent

OpenClaw starts with an onboarding flow — it asks a few questions and uses your answers to shape its personality and working style. Everything it learns gets written to a file called `SOUL.md` in your workspace.

> **Key idea:** the agent's behaviour is defined by durable files, not hardcoded logic. Change the files, change the agent.

### ⌨️ Open the TUI in your terminal

```bash
openclaw
```

Answer the onboarding questions — name, vibe, working style. Then come back and run the next cell.

In [ ]:
# Print the SOUL.md your answers created
workspace = subprocess.check_output(["openclaw", "workspace", "path"]).decode().strip()
soul_path = pathlib.Path(workspace) / "SOUL.md"
print(f"Workspace: {workspace}\n")
print(soul_path.read_text())

That file is the agent's identity. You can edit it directly — the agent picks up changes on the next message.

```
~/.openclaw/workspace/
├── SOUL.md     ← personality and working style  (you just saw this)
├── USER.md     ← what the agent knows about you
├── AGENTS.md   ← how to coordinate with other agents
└── skills/     ← reusable capabilities (we'll build one later)
```

---
## Section 2 — The Typing App

### 💬 Give the agent the repo in the TUI

```
I have a Python typing speed app. The source is at: /home/user/git/open_type_faster
Can you explore it and give me a one-paragraph summary of how it works?
```

Watch how it reads files and builds context — this is the **observe** phase of the agent loop.

In [ ]:
repo_path = pathlib.Path(".").resolve()
print(f"Repo: {repo_path}\n")
for f in sorted(repo_path.glob("*.py")):
    print(f"  {f.name}")

```
main.py    → CLI entry point
engine.py  → keystroke loop and session state
stats.py   → WPM, accuracy, suggestions  ← pure functions, easy to test
display.py → Rich terminal UI
history.py → JSON session persistence
words.py   → word lists by difficulty
config.py  → Theme and Config dataclasses
```

---
## Section 3 — Run the App and Spot the Bug

### ⌨️ Run the typing app in your terminal

```bash
python3 main.py
```

Type for 30 seconds. When the results screen appears, look closely.

> **Did you notice anything wrong?**

---

Let's run the test suite to see what it tells us.

In [ ]:
result = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:])

3 tests failing — all in `TestAccuracy`. The tests say exactly what *should* happen:

- `test_half_correct` → `_accuracy(5, 10)` should be `50.0`, got `0`
- `test_rounding` → `_accuracy(2, 3)` should be `66.7`, got `0`
- `test_error_session` → accuracy should be `80.0`, got `0`

Something in the accuracy calculation always returns 0. Let's give it to the agent.

---
## Section 4 — Let the Agent Find and Fix It

The agent has file tools — it can read source, run commands, and write edits.

### 💬 Paste this into the TUI

```
The test suite for the typing app is failing. Run the tests, read the relevant
source files, identify the root cause, and fix it.

Start with: python3 -m pytest tests/ -v --tb=short
```

Watch the loop:
1. **Run** the tests → see what's failing
2. **Read** the test → understand expected behaviour
3. **Read** `stats.py` → trace the bug to its source
4. **Fix** the minimal line
5. **Re-run** to confirm

This is the **think → act → observe → repeat** loop that makes it an agent, not a chatbot.

---
## Section 5 — Verify the Fix

In [ ]:
result = subprocess.run(
    ["python3", "-m", "pytest", "tests/", "-v"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
passed = "failed" not in result.stdout
print("\n" + ("✅ All tests passing" if passed else "❌ Some tests still failing"))

In [ ]:
diff = subprocess.run(["git", "diff"], capture_output=True, text=True, cwd=str(repo_path))
print(diff.stdout if diff.stdout else "(no uncommitted changes — agent already committed the fix)")

### ⌨️ Run the app again

```bash
python3 main.py
```

Accuracy should now show a real number.

---

One bug. Found and fixed autonomously. The agent ran the tests, read the code, reasoned about the root cause, and made the minimal change.

**This is what agents are for** — not answering questions, but doing developer work.

---
## Section 6 — Package It as a Reusable Skill

What the agent just did was ad-hoc. A **skill** makes that capability reusable — invoke it on any Python repo and the agent follows the same systematic approach.

### 💬 Ask the agent to create the skill

```
Package what you just did as a reusable skill called pytest-debugger.
It should work on any Python repo, not just this one.
```

Or create it directly:

In [ ]:
skill_dir = pathlib.Path(workspace) / "skills" / "pytest-debugger"
skill_dir.mkdir(parents=True, exist_ok=True)

skill_md = """# pytest-debugger

## Purpose
Investigate a failing Python test suite, identify the root cause of each
failure, and apply minimal fixes to make all tests pass.

## When to use
Invoke when given a repo path and told that tests are failing.

## Steps
1. Run `python3 -m pytest <path>/tests/ -v --tb=short` and capture output
2. For each FAILED test: read the test to understand expected behaviour
3. Read the source file referenced in the traceback
4. Identify the minimal line(s) causing the failure
5. Apply the fix — change only what is necessary
6. Re-run the tests to confirm the fix works
7. Report: file changed, line changed, what was wrong, what the fix was

## Rules
- Never rewrite functions — make the smallest possible edit
- Never skip a test — fix the code, not the test
- Always re-run after fixing to confirm
"""

(skill_dir / "skill.md").write_text(skill_md)
print("✅ pytest-debugger skill created")
print(skill_md)

In [ ]:
result = subprocess.run(["openclaw", "skills", "list"], capture_output=True, text=True)
print(result.stdout)

Now anyone can run:
```bash
openclaw skill pytest-debugger /path/to/any/python/repo
```
and get the same systematic debugging behaviour on any repo.

---
## Section 7 — Build a Typing Coach Agent

The app already tracks your session history — WPM, accuracy, mistyped words — in `~/.open_type_faster_history.json`.

Let's extend the agent to read that data and act as a personalised typing coach. We do it by adding a new section to `SOUL.md`.

In [ ]:
coaching_addition = """

## Typing Coach Mode
When asked to review typing performance:
1. Read ~/.open_type_faster_history.json
2. Look at the last 5 sessions: WPM trend, accuracy trend, backspace rate
3. Identify the single most impactful thing to improve
4. Give one concrete drill or technique — not a generic list of tips
5. Call out specific mistyped words if they appear across multiple sessions
"""

with soul_path.open("a") as f:
    f.write(coaching_addition)

print("✅ SOUL updated")
print(soul_path.read_text())

### ⌨️ Run the app a couple more times to build up some history

```bash
python3 main.py
```

Then ask the agent for coaching:

### 💬 Paste into the TUI

```
Review my typing session history and tell me the one thing I should focus on
to improve my speed. Be specific — look at the actual data.
```

The agent will read your real history file and give feedback based on your actual sessions — not generic advice.

---
## Section 8 — ClawHub

Everything you built today is local. ClawHub makes it shareable.

```
Local skill  →  openclaw skill publish  →  ClawHub registry
Anyone       →  openclaw skill install pytest-debugger  →  their workspace
```

The skill carries the SOUL context with it — whoever installs it gets the same behaviour you shaped, not a generic agent.

---

## What You Built

| Step | What happened |
|---|---|
| Onboarding | Agent learned your name, vibe, working style → wrote it to SOUL.md |
| Explore | Agent read a real codebase and understood its structure |
| Debug | Agent ran tests, traced a bug to one character, applied a minimal fix, re-verified |
| Skill | That debugging behaviour is now reusable on any Python repo |
| Coach | Agent reads your real session data and gives personalised improvement advice |

---

## Challenge — Pick a Track

| Track | What to do |
|---|---|
| **Bug Hunter** | Introduce your own bug into a copy of the repo, swap with a neighbour, fix each other's using the agent |
| **Skill Builder** | Extend `pytest-debugger` to write a `bug_report.md` after each fix (file, line, root cause, fix) |
| **Soul Shaper** | Edit SOUL.md to make the coach explain bugs as if teaching a junior developer — re-run and compare the output |

*Built something interesting? Share it and you may qualify for additional AMD Developer Cloud credits.*